# Demo instructor 01 · Analítica de ventas y construcción de modelos

            La dirección comercial de FraSoHome quiere priorizar dos decisiones: retener clientes con riesgo de inactividad y ordenar contactos de campaña según su probabilidad de conversión. Esta demo recorre el proceso completo de un proyecto de ML aplicado a ventas: carga de fuentes operativas, limpieza, ventanas temporales, ingeniería de características, preprocesado, entrenamiento y generación de artefactos para interpretabilidad.

            No se crean CSV nuevos. La fuente de verdad son los ficheros raw de `datos`.


In [1]:
from pathlib import Path
import sys
import warnings
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datos").exists() and (candidate / "escenarios").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado la raíz del proyecto FraSoHome.")

PROJECT_ROOT = find_project_root()
DEMO_DIR = PROJECT_ROOT / "escenarios" / "04-analitica-ventas-ml"
ARTIFACT_DIR = DEMO_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(DEMO_DIR))

from common.frasohome_ml import (
    build_campaign_dataset,
    build_churn_dataset,
    build_return_risk_dataset,
    clean_fact_for_ml,
    load_artifact,
    load_operational_tables,
    model_importance,
    prepare_crm,
    prepare_fact,
    save_artifact,
    score_deciles,
    train_binary_model,
    transformed_feature_names,
)

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: C:\repos\frasohome-demos


## 1. Carga y diagnóstico de fuentes

            Trabajamos con transacciones, CRM, campañas, navegación digital, feedback y tickets. En negocio, esto representa una vista incompleta pero realista del cliente: compra, relación, servicio y activación.


In [2]:
tables = load_operational_tables(PROJECT_ROOT)
summary = pd.DataFrame([
    {"tabla": name, "filas": df.shape[0], "columnas": df.shape[1]}
    for name, df in tables.items()
])
display(summary)


,tabla,filas,columnas
0,fact,4973,59
1,crm,103,20
2,productos,101,18
3,stock,1940,9
4,contactos,480,23
5,web,9030,22
6,feedback,850,15
7,tickets,560,20
8,costes,1829,21


## 2. Limpieza y tipado para ML

            La tabla integrada conserva problemas intencionales: fechas fuera de ventana, importes mixtos, devoluciones, IDs no normalizados y registros sin cliente. Primero tipamos y después aplicamos un filtro analítico 2024-2025, dejando trazabilidad de la reducción.


In [3]:
fact_typed = prepare_fact(tables["fact"])
fact = clean_fact_for_ml(fact_typed)
crm = prepare_crm(tables["crm"])

quality = pd.DataFrame({
    "métrica": ["filas raw", "filas ML", "clientes ML", "productos ML", "min raw", "max raw", "min ML", "max ML"],
    "valor": [
        len(fact_typed), len(fact), fact["customer_id_std"].nunique(), fact["product_id_std"].nunique(),
        fact_typed["event_dt"].min(), fact_typed["event_dt"].max(), fact["event_dt"].min(), fact["event_dt"].max()
    ],
})
display(quality)


,métrica,valor
0,filas raw,4973
1,filas ML,1237
2,clientes ML,117
3,productos ML,121
4,min raw,2017-10-02 00:00:00
5,max raw,2031-01-07 10:00:00
6,min ML,2024-01-02 03:21:03
7,max ML,2025-12-12 17:47:00


## 3. Caso 1: riesgo de inactividad a 90 días

            Cada fila es una fotografía de cliente. Usamos 180 días de histórico para construir variables y observamos los 90 días posteriores para etiquetar si el cliente queda inactivo. Así evitamos entrenar con información futura.


In [4]:
churn_df = build_churn_dataset(
    fact=fact,
    crm=crm,
    web=tables.get("web"),
    feedback=tables.get("feedback"),
    tickets=tables.get("tickets"),
    lookback_days=180,
    horizon_days=90,
    max_snapshots=8,
)
print("churn_df:", churn_df.shape)
display(churn_df["label_inactive_90d"].value_counts(normalize=True).rename("proporción").to_frame())
display(churn_df.head())


churn_df: (596, 62)


,proporción
label_inactive_90d,
0,0.526846
1,0.473154


,customer_id_std,first_purchase_dt,last_purchase_dt,frequency_docs,sales_lines,units_sold,monetary_gross,margin_gross,avg_discount_pct,pct_discounted_lines,avg_unit_price,n_categories,n_products,recency_days,customer_tenure_window_days,avg_order_value,gross_margin_rate,return_lines,amount_returned,units_returned,return_amount_rate,return_unit_rate,revenue_channel_online,revenue_channel_pos,online_revenue_share,revenue_cat_decoracion,revenue_cat_iluminacion,revenue_cat_muebles,revenue_cat_nan,revenue_cat_textil_hogar,last_channel,last_category,last_payment_method,last_store_id,tier_fidelizacion,points,estado_cliente,consentimiento_marketing,origen_alta,canal_preferido_declarado,ciudad,provincia,age_years,program_tenure_days,web_events,web_sessions,web_active_days,web_value_sum,web_product_views,web_searches,web_add_to_cart,feedback_count,avg_rating,avg_nps,pct_negative_feedback,service_tickets,avg_sla_hours,avg_csat,pct_high_priority_tickets,compensation_sum,label_inactive_90d,snapshot_date
0,C0003,2025-01-09 11:51:00,2025-01-09 11:51:00,1,3,5.0,1008.49,419.53,8.333333,0.333333,204.096667,3,3,22,0,1008.49,0.415998,0.0,0.0,0.0,0.0,0.0,0.00,1008.49,0.0,129.65,0.00,820.78,0.0,58.06,POS,Muebles,NaN,S002,Plata,2.073,Activo,YES,NaN,Mixto,Córdoba,Córdoba,0.000000,0.0,3.0,1.0,1.0,0.00,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,2025-02-01
1,C0004,2024-09-02 07:57:00,2024-09-02 07:57:00,1,1,1.0,331.35,137.92,30.000000,1.000000,473.350000,1,1,151,0,331.35,0.416237,0.0,0.0,0.0,0.0,0.0,331.35,0.00,1.0,0.00,0.00,331.35,0.0,0.00,ONLINE,Muebles,ContraReembolso,ONLINE,Oro,2073.000,Activo,0,importación,Mixto,Palma,Illes Balears,0.000000,0.0,4.0,1.0,1.0,0.00,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,2025-02-01
2,C0007,2024-09-07 13:57:13,2024-09-07 13:57:13,1,1,1.0,676.00,138.68,10.000000,1.000000,0.000000,1,1,146,0,676.00,0.205148,0.0,0.0,0.0,0.0,0.0,676.00,0.00,1.0,0.00,0.00,676.00,0.0,0.00,ONLINE,Muebles,Bizum,ONLINE,Bronce,1696.000,Activo,N,tienda,NaN,Granada,Granada,0.000000,312.0,10.0,2.0,2.0,866.86,4.0,2.0,1.0,1.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,2025-02-01
3,C0013,2025-01-09 23:40:01,2025-01-09 23:40:01,1,1,2.0,140.13,-56.55,15.000000,1.000000,82.000000,1,1,22,0,140.13,-0.403554,0.0,0.0,0.0,0.0,0.0,140.13,0.00,1.0,140.13,0.00,0.00,0.0,0.00,ONLINE,Decoración,BIZUM,ONLINE,Plata,6547.000,Activo,1,tienda,NaN,Madrid,Madrid,40.309377,2152.0,13.0,4.0,4.0,0.00,9.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,2025-02-01
4,C0015,2024-11-03 11:50:32,2024-11-03 11:50:32,1,4,5.0,1873.59,478.82,17.500000,0.750000,460.022500,2,4,89,0,1873.59,0.255563,0.0,0.0,0.0,0.0,0.0,1873.59,0.00,1.0,0.00,524.31,1349.28,0.0,0.00,ONLINE,Muebles,Tarjeta crédito,ONLINE,Plata,1587.000,Activo,sí,online,Mixto,Zaragoza,Zaragoza,54.568104,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,2025-02-01


In [5]:
CHURN_TARGET = "label_inactive_90d"
CHURN_EXCLUDED = ["customer_id_std", "snapshot_date", "first_purchase_dt", "last_purchase_dt"]
churn_result = train_binary_model(churn_df, CHURN_TARGET, CHURN_EXCLUDED, model_kind="rf", random_state=42)

display(pd.Series(churn_result["metrics"]).to_frame("valor"))
display(model_importance(churn_result["pipeline"], top_n=15))

churn_test = churn_result["X_test"].copy()
churn_test[CHURN_TARGET] = churn_result["y_test"].values
churn_score = churn_result["pipeline"].predict_proba(churn_result["X_test"])[:, 1]
display(score_deciles(churn_test, churn_score, CHURN_TARGET))


,valor
rows,596.000000
target_rate,0.473154
test_roc_auc,0.755515
test_average_precision,0.768213
test_f1,0.619048
test_accuracy,0.677852


,feature,importance,abs_value
0,points,0.062205,0.062205
1,margin_gross,0.035031,0.035031
2,avg_order_value,0.033205,0.033205
3,avg_sla_hours,0.031737,0.031737
4,ciudad_Córdoba,0.029915,0.029915
5,avg_unit_price,0.028762,0.028762
6,pct_high_priority_tickets,0.028633,0.028633
7,gross_margin_rate,0.028455,0.028455
8,monetary_gross,0.024707,0.024707
9,revenue_cat_muebles,0.023961,0.023961


,decile,n,avg_score,target_rate
9,10,15,0.813768,0.933333
8,9,15,0.648768,0.733333
7,8,15,0.565116,0.733333
6,7,15,0.509373,0.400000
5,6,14,0.448469,0.500000
4,5,15,0.409957,0.466667
3,4,15,0.371650,0.266667
2,3,15,0.326856,0.266667
1,2,15,0.263757,0.000000
0,1,15,0.188664,0.400000


## 4. Caso 2: propensión de conversión de campañas

            Ahora bajamos al contacto comercial. La etiqueta es conversión a 30 días y las variables combinan contexto del contacto con historial anterior del cliente. Excluimos resultados posteriores al contacto para evitar fuga de información.


In [6]:
campaign_df = build_campaign_dataset(
    contactos_raw=tables["contactos"],
    fact=fact,
    crm=crm,
    web=tables.get("web"),
    feedback=tables.get("feedback"),
    tickets=tables.get("tickets"),
    lookback_days=180,
)
print("campaign_df:", campaign_df.shape)
display(campaign_df["conversion_30d_flag"].value_counts(normalize=True).rename("proporción").to_frame())


campaign_df: (158, 84)


,proporción
conversion_30d_flag,
0,0.632911
1,0.367089


In [7]:
CAMPAIGN_TARGET = "conversion_30d_flag"
CAMPAIGN_EXCLUDED = [
    "contact_id", "campaign_id", "customer_id", "customer_id_std", "fecha_contacto", "contact_dt",
    "conversion_30d_flag", "order_id_conversion_30d", "ticket_id_conversion_30d", "fecha_conversion_30d",
    "revenue_30d_eur", "margen_30d_eur", "first_purchase_dt", "last_purchase_dt",
]
campaign_result = train_binary_model(campaign_df, CAMPAIGN_TARGET, CAMPAIGN_EXCLUDED, model_kind="rf", random_state=42)

display(pd.Series(campaign_result["metrics"]).to_frame("valor"))
display(model_importance(campaign_result["pipeline"], top_n=15))

campaign_test = campaign_result["X_test"].copy()
campaign_test[CAMPAIGN_TARGET] = campaign_result["y_test"].values
campaign_score = campaign_result["pipeline"].predict_proba(campaign_result["X_test"])[:, 1]
display(score_deciles(campaign_test, campaign_score, CAMPAIGN_TARGET))


,valor
rows,158.000000
target_rate,0.367089
test_roc_auc,0.688000
test_average_precision,0.646215
test_f1,0.451613
test_accuracy,0.575000


,feature,importance,abs_value
0,customer_tenure_window_days,0.058811,0.058811
1,revenue_channel_pos,0.056124,0.056124
2,margin_gross,0.033939,0.033939
3,gross_margin_rate,0.031391,0.031391
4,frequency_docs,0.031252,0.031252
5,avg_unit_price,0.026211,0.026211
6,avg_nps,0.025555,0.025555
7,monetary_gross,0.024387,0.024387
8,revenue_cat_textil_hogar,0.024125,0.024125
9,units_sold,0.019921,0.019921


,decile,n,avg_score,target_rate
9,10,4,0.692004,1.00
8,9,4,0.613925,0.50
7,8,4,0.545814,0.00
6,7,4,0.522159,0.25
5,6,4,0.478068,0.50
4,5,4,0.428589,0.25
3,4,4,0.364254,1.00
2,3,4,0.328565,0.00
1,2,4,0.278712,0.25
0,1,4,0.216948,0.00


## 5. Guardado del artefacto

            Este artefacto será la entrada de la sesión de interpretabilidad. Contiene datasets, modelos, columnas, métricas y particiones de entrenamiento/test.


In [8]:
artifact = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source": "CSV operativos FraSoHome 2024-2025",
    "churn": {"dataset": churn_df, "target": CHURN_TARGET, "excluded": CHURN_EXCLUDED, "result": churn_result},
    "campaign": {"dataset": campaign_df, "target": CAMPAIGN_TARGET, "excluded": CAMPAIGN_EXCLUDED, "result": campaign_result},
}
path = save_artifact(artifact, ARTIFACT_DIR / "sales_models_artifact.joblib")
print("Artefacto guardado:", path)


Artefacto guardado: C:\repos\frasohome-demos\escenarios\04-analitica-ventas-ml\artifacts\sales_models_artifact.joblib
